# 02 — CNN Model Training

Training experiments for the CNN-based deepfake audio detector.

This notebook walks through:
1. Dataset loading and class distribution visualisation
2. Model architecture summary
3. Full training loop with mixed-precision and early stopping
4. Live loss-curve plotting (train_loss and val_loss per epoch)
5. Validation metrics — accuracy, AUC, EER
6. Confusion matrix visualisation
7. Feature visualisation — sample mel-spectrograms for genuine vs synthetic
8. Checkpoint saving


In [ ]:
import sys
sys.path.insert(0, '..')

import warnings
warnings.filterwarnings('ignore')

import torch
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from IPython.display import display, clear_output

from src.models.cnn_detector import CNNDetector
from src.models.model_utils import get_device, save_checkpoint, count_parameters
from src.utils.config_loader import load_config

cfg = load_config('../configs/model_config.yaml')
device = get_device()
print(f'Device: {device}')
print(f'Config loaded: {list(cfg.keys())}')


## 1. Dataset Loading & Class Distribution


In [ ]:
import os
from pathlib import Path
from src.data.dataset_loader import DatasetLoader, DatasetType, Split
from src.data.audio_preprocessor import AudioPreprocessor
from src.data.feature_extractor import FeatureExtractor, FeatureType

# ── Configuration ─────────────────────────────────────────────────────────
DATA_DIR   = '../data/custom'   # <-- Change to your dataset directory
DATASET    = 'custom'           # asvspoof2019 | asvspoof5 | in_the_wild | custom
OUTPUT_DIR = '../models'

model_cfg = cfg.get('model', {})
train_cfg = cfg.get('training', {})

# ── Load dataset ───────────────────────────────────────────────────────────
if Path(DATA_DIR).exists():
    loader = DatasetLoader(
        dataset_type=DatasetType(DATASET),
        root_dir=DATA_DIR,
        split=Split.TRAIN,
    ).load()
    dist = loader.class_distribution()
    print(f'Dataset loaded: {dist}')

    # Class distribution bar chart
    fig, ax = plt.subplots(figsize=(5, 3))
    ax.bar(['Genuine', 'Synthetic'], [dist['genuine'], dist['synthetic']],
           color=['steelblue', 'tomato'])
    ax.set_title('Class Distribution')
    ax.set_ylabel('Sample Count')
    for i, v in enumerate([dist['genuine'], dist['synthetic']]):
        ax.text(i, v + 5, str(v), ha='center', fontweight='bold')
    plt.tight_layout()
    plt.show()
else:
    print(f'Data directory not found: {DATA_DIR}')
    print('Set DATA_DIR to a valid dataset path and re-run this cell.')
    loader = None


## 2. Feature Visualisation — Mel-Spectrograms


In [ ]:
import soundfile as sf
import librosa
import librosa.display

preprocessor    = AudioPreprocessor(target_sr=16_000, normalize=True)
feature_type    = FeatureType(model_cfg.get('input_feature', 'mel_spectrogram'))
feature_extractor = FeatureExtractor(feature_type=feature_type)

def show_spectrogram(sample, title=''):
    """Load, preprocess, and display a mel-spectrogram for one AudioSample."""
    waveform, sr = sf.read(str(sample.file_path), dtype='float32', always_2d=False)
    waveform, sr = preprocessor.process(waveform, sr)
    features = feature_extractor.extract(waveform)  # (C, H, W)
    spec = features[0]  # first channel
    plt.figure(figsize=(6, 3))
    librosa.display.specshow(spec, sr=sr, x_axis='time', y_axis='mel',
                             cmap='magma')
    plt.colorbar(format='%+2.0f dB')
    plt.title(title)
    plt.tight_layout()
    plt.show()

if loader and len(loader) > 0:
    all_samples = loader.samples
    genuine_samples  = [s for s in all_samples if s.label == 0][:3]
    synthetic_samples = [s for s in all_samples if s.label == 1][:3]

    for s in genuine_samples:
        show_spectrogram(s, title=f'Genuine — {s.file_path.name}')

    for s in synthetic_samples:
        show_spectrogram(s, title=f'Synthetic — {s.file_path.name}')
else:
    print('No samples available for visualisation.')


## 3. Build Model


In [ ]:
detector = CNNDetector(
    backbone=model_cfg.get('backbone', 'resnet34'),
    num_classes=model_cfg.get('num_classes', 2),
    pretrained=model_cfg.get('pretrained', True),
    dropout=model_cfg.get('dropout', 0.3),
    device=device,
).build()

model = detector._model
print(f'Backbone        : {model_cfg.get("backbone", "resnet34")}')
print(f'Trainable params: {count_parameters(model):,}')
print(f'Device          : {device}')


## 4. Build DataLoaders


In [ ]:
import random
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader

# Import the Dataset wrapper from the training script
sys.path.insert(0, '..')
from scripts.train import DeepfakeAudioDataset, _collate_skip_none

SEED       = train_cfg.get('seed', 42)
VAL_SPLIT  = train_cfg.get('val_split', 0.2)
BATCH_SIZE = train_cfg.get('batch_size', 32)

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if loader and len(loader) > 0:
    all_samples = loader.samples
    indices = list(range(len(all_samples)))
    random.shuffle(indices)
    n_val = max(1, int(len(indices) * VAL_SPLIT))
    val_idx, train_idx = indices[:n_val], indices[n_val:]
    train_samples = [all_samples[i] for i in train_idx]
    val_samples   = [all_samples[i] for i in val_idx]
    print(f'Train: {len(train_samples)} | Val: {len(val_samples)}')

    train_ds = DeepfakeAudioDataset(train_samples, preprocessor, feature_extractor)
    val_ds   = DeepfakeAudioDataset(val_samples,   preprocessor, feature_extractor)

    train_dl = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=0, collate_fn=_collate_skip_none, drop_last=True)
    val_dl   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=0, collate_fn=_collate_skip_none)
    print('DataLoaders ready.')
else:
    train_dl = val_dl = None
    print('No data available. DataLoaders not created.')


## 5. Training Loop with Live Loss Curves


In [ ]:
import torch.nn.functional as F
from sklearn.metrics import roc_auc_score

if train_dl is None:
    print('Skipping training — no DataLoaders.')
else:
    EPOCHS   = train_cfg.get('epochs', 20)
    LR       = train_cfg.get('learning_rate', 1e-4)
    WD       = train_cfg.get('weight_decay', 1e-4)
    PATIENCE = train_cfg.get('early_stopping_patience', 10)
    GRAD_CLIP = train_cfg.get('gradient_clip', 1.0)
    USE_AMP  = train_cfg.get('mixed_precision', True) and device == 'cuda'

    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=LR, weight_decay=WD)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)
    scaler    = torch.cuda.amp.GradScaler(enabled=USE_AMP)

    history = {'train_loss': [], 'val_loss': [], 'val_acc': [], 'val_auc': []}
    best_val_loss = float('inf')
    no_improve    = 0

    # Set up live plot figure
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))

    for epoch in range(1, EPOCHS + 1):
        # ── Train ──────────────────────────────────────────────────────────
        model.train()
        train_loss_sum, n_batches = 0.0, 0
        for features, labels in train_dl:
            if features is None:
                continue
            if features.shape[1] == 1:
                features = features.repeat(1, 3, 1, 1)
            features, labels = features.to(device), labels.to(device)
            optimizer.zero_grad()
            with torch.cuda.amp.autocast(enabled=USE_AMP):
                loss = criterion(model(features), labels)
            scaler.scale(loss).backward()
            if GRAD_CLIP > 0:
                scaler.unscale_(optimizer)
                nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
            scaler.step(optimizer)
            scaler.update()
            train_loss_sum += loss.item()
            n_batches += 1
        avg_train_loss = train_loss_sum / max(n_batches, 1)

        # ── Validate ───────────────────────────────────────────────────────
        model.eval()
        val_loss_sum, n_val_batches = 0.0, 0
        all_labels, all_probs, all_preds = [], [], []
        with torch.no_grad():
            for features, labels in val_dl:
                if features is None:
                    continue
                if features.shape[1] == 1:
                    features = features.repeat(1, 3, 1, 1)
                features, labels = features.to(device), labels.to(device)
                with torch.cuda.amp.autocast(enabled=USE_AMP):
                    logits = model(features)
                    loss   = criterion(logits, labels)
                probs = F.softmax(logits, dim=1).cpu().numpy()
                all_labels.extend(labels.cpu().tolist())
                all_probs.extend(probs[:, 1].tolist())
                all_preds.extend(probs.argmax(axis=1).tolist())
                val_loss_sum += loss.item()
                n_val_batches += 1
        avg_val_loss = val_loss_sum / max(n_val_batches, 1)
        val_acc  = sum(p == l for p, l in zip(all_preds, all_labels)) / max(len(all_labels), 1)
        try:
            val_auc = roc_auc_score(all_labels, all_probs)
        except Exception:
            val_auc = 0.5

        scheduler.step()
        history['train_loss'].append(avg_train_loss)
        history['val_loss'].append(avg_val_loss)
        history['val_acc'].append(val_acc)
        history['val_auc'].append(val_auc)

        print(f'Epoch {epoch:3d}/{EPOCHS} | '
              f'train_loss={avg_train_loss:.4f} | '
              f'val_loss={avg_val_loss:.4f} | '
              f'val_acc={val_acc:.4f} | '
              f'val_auc={val_auc:.4f}')

        # ── Live plot update ───────────────────────────────────────────────
        clear_output(wait=True)
        epochs_range = range(1, epoch + 1)
        axes[0].cla()
        axes[0].plot(epochs_range, history['train_loss'], label='Train Loss')
        axes[0].plot(epochs_range, history['val_loss'],   label='Val Loss')
        axes[0].set_title('Loss Curves')
        axes[0].set_xlabel('Epoch')
        axes[0].set_ylabel('Loss')
        axes[0].legend()
        axes[1].cla()
        axes[1].plot(epochs_range, history['val_acc'], label='Val Accuracy', color='steelblue')
        axes[1].plot(epochs_range, history['val_auc'], label='Val AUC',      color='darkorange')
        axes[1].set_title('Validation Metrics')
        axes[1].set_xlabel('Epoch')
        axes[1].set_ylabel('Score')
        axes[1].set_ylim(0, 1)
        axes[1].legend()
        plt.suptitle(f'Epoch {epoch}/{EPOCHS}')
        plt.tight_layout()
        display(fig)

        # ── Early stopping & best checkpoint ──────────────────────────────
        if avg_val_loss < best_val_loss:
            best_val_loss = avg_val_loss
            no_improve = 0
            save_checkpoint(
                model=model, optimizer=optimizer, epoch=epoch,
                metrics={'val_loss': avg_val_loss, 'val_accuracy': val_acc, 'val_auc': val_auc},
                checkpoint_dir=OUTPUT_DIR,
                filename='best_checkpoint.pt',
            )
            print(f'  ✓ New best model saved (val_loss={avg_val_loss:.4f})')
        else:
            no_improve += 1
            if PATIENCE > 0 and no_improve >= PATIENCE:
                print(f'Early stopping after {no_improve} epochs without improvement.')
                break

    plt.close(fig)
    print(f'Training complete. Best val_loss={best_val_loss:.4f}')


## 6. Confusion Matrix


In [ ]:
import seaborn as sns
from sklearn.metrics import confusion_matrix, classification_report

if val_dl is not None and len(all_labels) > 0:
    cm = confusion_matrix(all_labels, all_preds)
    fig, ax = plt.subplots(figsize=(5, 4))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=['Genuine', 'Synthetic'],
                yticklabels=['Genuine', 'Synthetic'], ax=ax)
    ax.set_title('Confusion Matrix (final epoch)')
    ax.set_ylabel('Actual')
    ax.set_xlabel('Predicted')
    plt.tight_layout()
    plt.show()
    print(classification_report(all_labels, all_preds,
                                 target_names=['Genuine', 'Synthetic']))
else:
    print('No predictions available for confusion matrix.')


## 7. Equal Error Rate (EER)


In [ ]:
from sklearn.metrics import roc_curve

if val_dl is not None and len(all_labels) > 0 and len(set(all_labels)) == 2:
    fpr, tpr, thresholds = roc_curve(all_labels, all_probs)
    fnr = 1 - tpr
    eer_idx = np.nanargmin(np.abs(fpr - fnr))
    eer = (fpr[eer_idx] + fnr[eer_idx]) / 2
    print(f'EER: {eer:.4f} ({eer*100:.2f}%)')

    fig, ax = plt.subplots(figsize=(6, 4))
    ax.plot(fpr, tpr, label=f'ROC curve (AUC={val_auc:.3f})')
    ax.plot([0, 1], [0, 1], 'k--', alpha=0.5)
    ax.scatter(fpr[eer_idx], tpr[eer_idx], color='red', zorder=5,
               label=f'EER = {eer:.4f}')
    ax.set_xlabel('False Positive Rate')
    ax.set_ylabel('True Positive Rate')
    ax.set_title('ROC Curve')
    ax.legend()
    plt.tight_layout()
    plt.show()
else:
    print('Cannot compute EER — predictions not available or only one class in validation set.')
